In [12]:
import cvxpy as cp
import numpy as np
import pandas as pd

# load what we just saved (or use variables directly)
mu = np.load("mu_annual.npy")          # shape (n,)
cov = np.load("cov_annual2.npy")        # shape (n, n)
tickers = pd.read_csv("tickers.csv", header=None)[0].tolist()

def max_return_under_vol(mu, cov, max_vol, max_weight=0.25,
                         sector_to_indices=None, min_sector_weight=0.01):
    """
    Maximize portfolio expected return: mu^T w
    subject to:
        sqrt(w^T cov w) <= max_vol
        sum(w) = 1
        0 <= w_i <= max_weight
    """
    n = len(mu)
    w = cp.Variable(n)

    # portfolio variance

    port_var = cp.quad_form(w, cov)

    # objective: maximize expected annual return
    objective = cp.Maximize(mu @ w)

    
    constraints = [
        cp.sum(w) == 1,
        w >= 0,
        w <= max_weight,
        port_var <= max_vol**2
    ]

    # ---- Sector constraints: at least X weight per sector ----
    if sector_to_indices is not None:
        for sector, indices in sector_to_indices.items():
            constraints.append(cp.sum(w[indices]) >= min_sector_weight)

    prob = cp.Problem(objective, constraints)
    prob.solve()

    if (w.value is None) or (prob.status not in ["optimal", "optimal_inaccurate"]):
        raise RuntimeError(f"Optimization failed: {prob.status}")

    w_opt = np.array(w.value).flatten()
    port_return = float(mu @ w_opt)
    port_vol = float(np.sqrt(w_opt @ cov @ w_opt))

    return w_opt, port_return, port_vol

In [13]:
## Portfolio Profiles

risk_limits = {
    "Aggressive":   0.45,   # 45% annual volatility
    "Balanced":     0.25,   # 25%
    "Conservative": 0.18    # 18%
}

for profile, max_vol in risk_limits.items():
    w, ret, vol = max_return_under_vol(mu, cov, max_vol=max_vol, max_weight=0.25)

    # Align tickers to the length of weights to avoid mismatched-length errors
    tickers_aligned = tickers[:len(w)]

    df = pd.DataFrame({
        "Ticker": tickers_aligned,
        "Weight": w
    }).sort_values("Weight", ascending=False)

    print("\n==========================================")
    print(profile.upper())
    print("==========================================")
    print(f"Max Expected Return: {ret * 100:.2f}%")
    print(f"Portfolio Vol:       {vol * 100:.2f}% (limit = {max_vol * 100:.1f}%)\n")

    print(df[df["Weight"] > 0.001])



AGGRESSIVE
Max Expected Return: 67.92%
Portfolio Vol:       28.41% (limit = 45.0%)

   Ticker  Weight
22    NEM    0.25
6     CAT    0.25
10    CVX    0.25
17    LIN    0.25

BALANCED
Max Expected Return: 63.99%
Portfolio Vol:       25.00% (limit = 25.0%)

   Ticker    Weight
22    NEM  0.250000
10    CVX  0.250000
17    LIN  0.185025
30      V  0.174409
6     CAT  0.140567

CONSERVATIVE
Max Expected Return: 49.52%
Portfolio Vol:       18.00% (limit = 18.0%)

   Ticker    Weight
30      V  0.228710
10    CVX  0.189068
31   WELL  0.188506
17    LIN  0.145404
22    NEM  0.129096
1    AAPL  0.071708
6     CAT  0.047508
